# 03 — Feature Engineering & Point-in-Time Processing

> **Mục tiêu**: Tiếp nhận toàn bộ các bảng dữ liệu thô (Raw Tables) từ generator, xây dựng hệ thống đặc trưng Point-in-Time strictly $< T$ không phụ thuộc vào nhãn hay thông tin hậu nghiệm, kiểm định rò rỉ dữ liệu (Anti-Leakage), thực hiện phân chia tập dữ liệu an toàn (Entity-Safe & Temporal Split) và đóng gói trọn bộ artifacts cho quá trình huấn luyện mô hình.
>
> **Input**: 9 bảng normalized CSV trong `fraud_data_generator_v2/output_training_raw/merged/` và `dataset_manifest.json`.
>
> **Output**: `feature_matrix_raw.parquet`, `train.parquet`, `validation.parquet`, `test.parquet`, `feature_registry.csv`, `split_manifest.json` trong thư mục `data/processed/`.

---

## Table of Contents

1. [Mục tiêu và Data Contract](#1-mục-tiêu-và-data-contract)
2. [Đọc Dữ liệu Đầu vào & Kiểm tra Khởi tạo](#2-đọc-dữ-liệu-đầu-vào--kiểm-tra-khởi-tạo)
3. [Chuẩn hóa Dữ liệu Thô (Data Normalization)](#3-chuẩn-hóa-dữ-liệu-thô-data-normalization)
4. [Hợp nhất Ngữ cảnh Thực thể (Context Joins & Cardinality Audit)](#4-hợp-nhất-ngữ-cảnh-thực-thể-context-joins--cardinality-audit)
5. [Đặc trưng Số tiền & Hạn mức (Financial & Limit Features)](#5-đặc-trưng-số-tiền--hạn-mức-financial--limit-features)
6. [Đặc trưng Lịch sử & Tần suất Cuộn (Point-in-Time Velocity Features)](#6-đặc-trưng-lịch-sử--tần-suất-cuộn-point-in-time-velocity-features)
7. [Đặc trưng Thời gian & Chu kỳ Tuần hoàn (Cyclical Temporal Features)](#7-đặc-trưng-thời-gian--chu-kỳ-tuần-hoàn-cyclical-temporal-features)
8. [Đặc trưng Thiết bị & Vị trí (Device & Geo-Location Dynamics)](#8-đặc-trưng-thiết-bị--vị-trí-device--geo-location-dynamics)
9. [Đặc trưng Người Thụ Hưởng (Beneficiary Dynamics)](#9-đặc-trưng-người-thụ-hưởng-beneficiary-dynamics)
10. [Đặc trưng Xác thực & Biến động Nhạy cảm (Auth & Account Changes)](#10-đặc-trưng-xác-thực--biến-động-nhạy-cảm-auth--account-changes)
11. [Đặc trưng Chuỗi Tương tác Phức hợp (Composite Attack Sequences)](#11-đặc-trưng-chuỗi-tương-tác-phức-hợp-composite-attack-sequences)
12. [Chính sách Missing & Biến đổi Duration](#12-chính-sách-missing--biến-đổi-duration)
13. [Kiểm toán Rò rỉ Dữ liệu & Danh mục Cấm (Leakage & Deny-List Audit)](#13-kiểm-toán-rò-rỉ-dữ-liệu--danh-mục-cấm-leakage--deny-list-audit)
14. [Kiểm tra Toàn diện Feature Matrix (Matrix Validation Assertions)](#14-kiểm-tra-toàn-diện-feature-matrix-matrix-validation-assertions)
15. [Phân chia Tập Dữ liệu An toàn (Entity-Safe & Temporal Split)](#15-phân-chia-tập-dữ-liệu-an-toàn-entity-safe--temporal-split)
16. [Bảng Đăng ký Hợp đồng Đặc trưng (Feature Contract Registry)](#16-bảng-đăng-ký-hợp-đồng-đặc-trưng-feature-contract-registry)
17. [Đóng gói & Xuất Artifacts Bàn giao (Artifacts Export)](#17-đóng-gói--xuất-artifacts-bàn-giao-artifacts-export)

---
<a id="1"></a>
## 1. Mục tiêu và Data Contract

### 1.1 Nguyên tắc Point-in-Time & Anti-Leakage
- **Grain**: Một dòng duy nhất tương ứng với một `transaction_id`.
- **Thời điểm quan sát (Point-in-time cutoff)**: $T = \text{transaction\_at}$.
- **Causality Constraint**: Toàn bộ các đặc trưng lịch sử (tần suất, tổng tiền, số lần đăng nhập sai, đổi mật khẩu...) chỉ được tính trên các sự kiện phát sinh trước $T$ ($\text{event\_time} < T$).
- **Không tái sử dụng feature có sẵn**: Tuyệt đối không đọc các cột tính sẵn trong `transaction_features.csv` hay chạy `rebuild_features.py`; toàn bộ logic được trích xuất từ dữ liệu thô.
- **Không sử dụng thông tin hậu nghiệm**: Cấm toàn bộ các trường về kịch bản (`scenario_code`, `scenario_hint`, `_SCN_`), mạng lưới mule synthetic (`mule_cluster_id`), cảnh báo hậu nghiệm (`alert_id`, `case_id`, `investigation_outcome`).


In [ ]:
import json
import os
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

# Thiết lập đường dẫn import và thư mục đầu ra
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = REPO_ROOT / "notebooks" / "src"
if SRC_DIR.exists() and str(SRC_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_DIR.resolve()))

from viz_utils import setup, clean_ax, PALETTE
setup()

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

# ─────────────────────────────────────────────────────────────────────────────
# ĐỊNH NGHĨA DATA CONTRACT
# ─────────────────────────────────────────────────────────────────────────────
TARGET_COL = "target_fraud"

AUDIT_COLS = [
    "transaction_id",
    "account_id",
    "customer_id",
    "simulation_run_id",
    "sample_weight",
    "hard_negative",
    "event_id",
    "scenario_code", 
    "label_scope",
    "amount_num",
    "beneficiary_id",
    "txn_time_utc",
]

DENY_COLS = [
    "scenario_code",
    "scenario_hint",
    "mule_cluster_id",
    "event_id",
    "fraud_label",
    "investigation_outcome",
    "population",
    "entity_role",
    "label_scope"
]

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Đã khởi tạo Data Contract:")
print(f"  - Target Column : {TARGET_COL}")
print(f"  - Audit Columns : {AUDIT_COLS}")
print(f"  - Deny Columns  : {DENY_COLS}")
print(f"  - Processed Dir : {PROCESSED_DIR.resolve()}")

---
<a id="2"></a>
## 2. Đọc Dữ liệu Đầu vào & Kiểm tra Khởi tạo

### Mục tiêu & Kiểm tra Ban đầu
- Đọc `dataset_manifest.json` làm nguồn kiểm chứng số dòng thực tế.
- Nạp đầy đủ **9 bảng dữ liệu thô**: `transactions`, `accounts`, `customers`, `devices`, `login_sessions`, `beneficiaries`, `account_change_events`, `auth_events`, `scenario_event_entities` (chỉ dùng để lấy target & audit).
- **Assertions khởi tạo**:
  1. Kiểm tra tính duy nhất của `transaction_id`.
  2. Kiểm tra tính toàn vẹn của nhãn mục tiêu (`target_fraud` $\in \{0, 1\}$).
  3. Kiểm tra tính hợp lệ của số lượng bản ghi so với manifest.


In [ ]:
DATA_DIR = REPO_ROOT / "fraud_data_generator_v2" / "output_training_raw" / "merged"
manifest_path = DATA_DIR / "dataset_manifest.json"
assert manifest_path.exists(), f"Không tìm thấy manifest: {manifest_path}"

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

expected_txn_count = manifest["row_counts"]["transactions.csv"]

# 1. Nạp 9 bảng dữ liệu thô (đọc dạng string để kiểm soát kiểu ép)
print("Đang nạp các bảng dữ liệu thô từ:", DATA_DIR)
t0 = time.time()

raw_transactions = pd.read_csv(DATA_DIR / "transactions.csv", dtype=str)
raw_accounts = pd.read_csv(DATA_DIR / "accounts.csv", dtype=str)
raw_customers = pd.read_csv(DATA_DIR / "customers.csv", dtype=str)
raw_devices = pd.read_csv(DATA_DIR / "devices.csv", dtype=str)
raw_sessions = pd.read_csv(DATA_DIR / "login_sessions.csv", dtype=str)
raw_beneficiaries = pd.read_csv(DATA_DIR / "beneficiaries.csv", dtype=str)
raw_change_events = pd.read_csv(DATA_DIR / "account_change_events.csv", dtype=str)
raw_auth_events = pd.read_csv(DATA_DIR / "auth_events.csv", dtype=str)
raw_bridge = pd.read_csv(DATA_DIR / "scenario_event_entities.csv", dtype=str)

print(f"✓ Đã nạp thành công 9 bảng dữ liệu thô trong {time.time() - t0:.2f} giây.")

# 2. Assertions khởi tạo bắt buộc
assert len(raw_transactions) == expected_txn_count, (
    f"Lỗi số dòng: transactions có {len(raw_transactions):,}, kỳ vọng {expected_txn_count:,}"
)
assert raw_transactions["transaction_id"].is_unique, "Lỗi: transaction_id không duy nhất!"

# 3. Trích xuất nhãn từ scenario_event_entities (Chỉ lấy grain transaction)
bridge_txn = raw_bridge[raw_bridge["entity_type"] == "transaction"].copy()
assert bridge_txn["entity_id"].is_unique, "Lỗi: entity_id trong bridge bị trùng lặp!"
bridge_map = bridge_txn.set_index("entity_id")

raw_transactions["target_fraud"] = raw_transactions["transaction_id"].map(bridge_map["target_fraud"]).fillna("0").astype(int)
raw_transactions["hard_negative"] = raw_transactions["transaction_id"].map(bridge_map["hard_negative"]).fillna("0").astype(int)
raw_transactions["sample_weight"] = raw_transactions["transaction_id"].map(bridge_map["sample_weight"]).fillna("1.0").astype(float)

assert set(raw_transactions["target_fraud"].unique()).issubset({0, 1}), "Lỗi: Target chứa giá trị ngoài {0, 1}!"

print(f"✓ Kiểm tra khởi tạo thành công:")
print(f"  - Tổng số giao dịch : {len(raw_transactions):,} dòng")
print(f"  - Confirmed Fraud   : {(raw_transactions['target_fraud'] == 1).sum():,} ({raw_transactions['target_fraud'].mean():.2%})")
print(f"  - Hard-Negative     : {(raw_transactions['hard_negative'] == 1).sum():,} ({raw_transactions['hard_negative'].mean():.2%})")


---
<a id="3"></a>
## 3. Chuẩn hóa Dữ liệu Thô (Data Normalization)

### Mục tiêu & Quy tắc Chuẩn hóa
- **Thời gian**: Chuẩn hóa datetime về UTC và chuyển đổi chính xác sang giờ địa phương Việt Nam (`Asia/Ho_Chi_Minh` — UTC+7). Tạo các cột `txn_time_utc` và `txn_time_local`.
- **Số học**: Ép kiểu các trường tài chính (`amount`, `balance_before`, `balance_after`, `limits`) sang kiểu số thực `float64`.
- **Kiểm tra tính hợp lệ**: Khẳng định không có `amount < 0`, `balance_before < 0` hay `effective_limit <= 0`.
- **Dữ liệu chuỗi**: Chuẩn hóa chuỗi rỗng `""` hoặc khoảng trắng thành `NaN` (Missing Value).
- **Không tự động điền Missing**: Chưa thực hiện impute mean/median ở bước này để tránh rò rỉ dữ liệu phân phối trước khi split.


In [ ]:
# Helper chuẩn hóa boolean
def parse_bool(series):
    return series.astype(str).str.lower().isin(["true", "1", "1.0"]).astype(int)

# 1. Chuẩn hóa bảng transactions
df_txns = raw_transactions.copy()
df_txns["txn_time_utc"] = pd.to_datetime(df_txns["transaction_at"], utc=True)
df_txns["txn_time_local"] = df_txns["txn_time_utc"].dt.tz_convert("Asia/Ho_Chi_Minh")

df_txns["amount_num"] = pd.to_numeric(df_txns["amount"], errors="coerce")
df_txns["bal_before_num"] = pd.to_numeric(df_txns["balance_before"], errors="coerce")
df_txns["bal_after_num"] = pd.to_numeric(df_txns["balance_after"], errors="coerce")

assert (df_txns["amount_num"] >= 0).all(), "Lỗi: Phát hiện amount âm!"
assert (df_txns["bal_before_num"] >= 0).all(), "Lỗi: Phát hiện balance_before âm!"

# 2. Chuẩn hóa bảng accounts
df_accs = raw_accounts.copy()
df_accs["single_txn_limit_num"] = pd.to_numeric(df_accs["single_txn_limit"], errors="coerce")
df_accs["daily_transfer_limit_num"] = pd.to_numeric(df_accs["daily_transfer_limit"], errors="coerce")
df_accs["average_balance_num"] = pd.to_numeric(df_accs["average_balance"], errors="coerce")
df_accs["effective_limit"] = np.minimum(df_accs["single_txn_limit_num"], df_accs["daily_transfer_limit_num"])
df_accs["open_date_utc"] = pd.to_datetime(df_accs["open_date"], utc=True)

# 3. Chuẩn hóa bảng login_sessions
df_sess = raw_sessions.copy()
df_sess["login_at_utc"] = pd.to_datetime(df_sess["login_at"], utc=True)
df_sess["is_new_location_int"] = parse_bool(df_sess["is_new_location"])
df_sess["vpn_flag_int"] = parse_bool(df_sess["vpn_flag"])
df_sess["proxy_flag_int"] = parse_bool(df_sess["proxy_flag"])

# 4. Chuẩn hóa bảng devices
df_devs = raw_devices.copy()
df_devs["is_emulator_int"] = parse_bool(df_devs["is_emulator"])
df_devs["is_rooted_int"] = parse_bool(df_devs["is_rooted_or_jailbroken"])
df_devs["first_seen_at_utc"] = pd.to_datetime(df_devs["first_seen_at"], utc=True)

# 5. Chuẩn hóa bảng beneficiaries
df_bens = raw_beneficiaries.copy()
df_bens["bene_added_at_utc"] = pd.to_datetime(df_bens["added_at"], utc=True)

# 6. Chuẩn hóa bảng account_change_events
df_changes = raw_change_events.copy()
df_changes["changed_at_utc"] = pd.to_datetime(df_changes["changed_at"], utc=True)

# 7. Chuẩn hóa bảng auth_events
df_auths = raw_auth_events.copy()
df_auths["auth_at_utc"] = pd.to_datetime(df_auths["auth_at"], utc=True)

print("✓ Chuẩn hóa thành công toàn bộ các trường thời gian, số học và boolean trên 7 bảng thực thể.")


---
<a id="4"></a>
## 4. Hợp nhất Ngữ cảnh Thực thể (Context Joins & Cardinality Audit)

### Mục tiêu & Quy tắc Join An Toàn
- Hợp nhất `transactions` với các bảng ngữ cảnh: `accounts`, `customers`, `login_sessions`, `devices`, `beneficiaries`.
- **Cardinality Audit**:
  - `transactions` $\longrightarrow$ `accounts`: Many-to-One.
  - `transactions` $\longrightarrow$ `customers`: Many-to-One.
  - `transactions` $\longrightarrow$ `login_sessions`: Many-to-One (Nullable).
  - `transactions` $\longrightarrow$ `devices`: Many-to-One (Nullable).
  - `transactions` $\longrightarrow$ `beneficiaries`: Many-to-One (Nullable).
- **Assertions**: Tuyệt đối không làm tăng hoặc giảm số dòng của bảng transactions gốc.


In [ ]:
# Helper kiểm tra khóa duy nhất
def assert_unique_key(frame, key, table_name):
    null_cnt = frame[key].isna().sum()
    dup_cnt = frame[key].duplicated().sum()
    assert null_cnt == 0, f"Lỗi {table_name}.{key}: {null_cnt} khóa NULL"
    assert dup_cnt == 0, f"Lỗi {table_name}.{key}: {dup_cnt} khóa trùng lặp"

# 1. Khẳng định khóa chính của các bảng ngữ cảnh
assert_unique_key(df_accs, "account_id", "accounts")
assert_unique_key(df_customers := raw_customers.copy(), "customer_id", "customers")
assert_unique_key(df_sess, "session_id", "login_sessions")
assert_unique_key(df_devs, "device_id", "devices")
assert_unique_key(df_bens, "beneficiary_id", "beneficiaries")

# 2. Join Accounts
base_df = df_txns.merge(
    df_accs[["account_id", "account_type", "account_currency", "status", "effective_limit", "average_balance_num", "open_date_utc"]],
    on="account_id", how="left", validate="many_to_one"
)
assert len(base_df) == expected_txn_count, "Lỗi join accounts làm thay đổi số dòng!"

# 3. Join Customers
base_df = base_df.merge(
    df_customers[["customer_id", "customer_segment", "kyc_level", "base_risk_level", "occupation_group", "province"]],
    on="customer_id", how="left", validate="many_to_one"
)
assert len(base_df) == expected_txn_count, "Lỗi join customers làm thay đổi số dòng!"

# 4. Join Sessions
base_df = base_df.merge(
    df_sess[["session_id", "login_at_utc", "is_new_location_int", "vpn_flag_int", "proxy_flag_int", "country"]],
    on="session_id", how="left", validate="many_to_one"
)
assert len(base_df) == expected_txn_count, "Lỗi join sessions làm thay đổi số dòng!"

# 5. Join Devices
base_df = base_df.merge(
    df_devs[["device_id", "device_type", "os", "trust_status", "is_emulator_int", "is_rooted_int", "first_seen_at_utc"]],
    on="device_id", how="left", validate="many_to_one"
)
assert len(base_df) == expected_txn_count, "Lỗi join devices làm thay đổi số dòng!"

# 6. Join Beneficiaries
base_df = base_df.merge(
    df_bens[["beneficiary_id", "beneficiary_bank", "bene_added_at_utc"]],
    on="beneficiary_id", how="left", validate="many_to_one"
)
assert len(base_df) == expected_txn_count, "Lỗi join beneficiaries làm thay đổi số dòng!"
assert base_df["transaction_id"].is_unique, "Lỗi: transaction_id không còn duy nhất sau joins!"

print(f"✓ Hoàn tất Join Ngữ Cảnh: {base_df.shape[0]:,} dòng × {base_df.shape[1]} cột. Bảo toàn 100% grain giao dịch.")


---
<a id="5"></a>
## 5. Đặc trưng Số tiền & Hạn mức (Financial & Limit Features)

### Mục tiêu & Công thức
- `log_amount = log1p(amount)`.
- `amount_to_limit_ratio = amount / effective_limit`.
- `amount_to_pre_txn_balance_ratio = amount / balance_before`.
- `is_high_limit_usage` và `is_high_balance_drain` đánh dấu mức sử dụng từ 80% trở lên.

Các phép chia dùng `0.0` khi mẫu số không hợp lệ. Data-quality của hạn mức thuộc Notebook 01; Notebook 03 không tạo thêm cờ trung gian nếu chúng không đi vào model.

In [ ]:
# 1. Feature tài chính trực tiếp; không tạo bản sao/cờ trung gian không được model sử dụng.
base_df["log_amount"] = np.log1p(base_df["amount_num"])
base_df["amount_to_limit_ratio"] = np.where(
    base_df["effective_limit"] > 0,
    base_df["amount_num"] / base_df["effective_limit"],
    0.0,
)
base_df["amount_to_pre_txn_balance_ratio"] = np.where(
    base_df["bal_before_num"] > 0,
    base_df["amount_num"] / base_df["bal_before_num"],
    0.0,
)

# 2. Cờ hành vi tài chính.
base_df["is_high_limit_usage"] = (base_df["amount_to_limit_ratio"] >= 0.8).astype(int)
base_df["is_high_balance_drain"] = (base_df["amount_to_pre_txn_balance_ratio"] >= 0.8).astype(int)

print("✓ Hoàn thành Mục 5: Đặc trưng Số tiền & Hạn mức")
print(f"  - Số GD rút >= 80% số dư: {base_df['is_high_balance_drain'].sum():,} ({base_df['is_high_balance_drain'].mean():.2%})")
print(f"  - Số GD chạm >= 80% hạn mức: {base_df['is_high_limit_usage'].sum():,} ({base_df['is_high_limit_usage'].mean():.2%})")

---
<a id="6"></a>
## 6. Đặc trưng Lịch sử & Tần suất Cuộn (Point-in-Time Velocity Features)

### Mục tiêu & Nguyên tắc Point-in-Time
- **Sắp xếp thời gian**: Dữ liệu bắt buộc phải sắp xếp theo thứ tự `["account_id", "txn_time_utc", "transaction_id"]`.
- **Causality strictly $< T$**: Giao dịch hiện tại $T$ **không bao giờ nằm trong cửa sổ rolling**. Giao dịch đầu tiên của tài khoản phải có:
  $$\text{prior\_txn\_count\_10m} = 0, \quad \text{prior\_txn\_count\_1h} = 0, \quad \text{prior\_txn\_count\_24h} = 0$$
- **Cửa sổ tính toán**:
  - Tần suất: `prior_txn_count_10m`, `prior_txn_count_1h`, `prior_txn_count_24h`.
  - Dòng tiền tích lũy: `prior_txn_amount_sum_10m`, `prior_txn_amount_sum_1h`, `prior_txn_amount_sum_24h`.
  - Khoảng cách thời gian: `time_since_previous_txn_minutes`.
  - Đường cơ sở chi tiêu: `historical_median_amount_30d`, `amount_to_historical_median_ratio`.
  - Đặc trưng phái sinh: `avg_amount_per_recent_txn`, `velocity_intensity`, `velocity_x_median_spike`, `is_first_transaction`.


In [ ]:
# 1. Sắp xếp chặt chẽ theo dòng thời gian tài khoản
base_df = base_df.sort_values(["account_id", "txn_time_utc", "transaction_id"]).reset_index(drop=True)
base_df["txn_time_int"] = base_df["txn_time_utc"].astype(np.int64) // 10**9

n_rows = len(base_df)
c_10m = np.zeros(n_rows, dtype=np.int32)
c_1h = np.zeros(n_rows, dtype=np.int32)
c_24h = np.zeros(n_rows, dtype=np.int32)
s_10m = np.zeros(n_rows, dtype=np.float64)
s_1h = np.zeros(n_rows, dtype=np.float64)
s_24h = np.zeros(n_rows, dtype=np.float64)
time_since_prev = np.full(n_rows, np.nan, dtype=np.float64)
hist_median_30d = np.full(n_rows, np.nan, dtype=np.float64)

# 2. Mỗi transaction chỉ nhìn thấy lịch sử có timestamp strictly < T.
# hist_end dùng side="left" nên loại toàn bộ transaction cùng timestamp, không phụ thuộc transaction_id.
t0 = time.time()
for _, group_indices in base_df.groupby("account_id", sort=False).groups.items():
    idx = group_indices.to_numpy()
    times = base_df.loc[idx, "txn_time_int"].to_numpy()
    amounts = base_df.loc[idx, "amount_num"].to_numpy()

    for i, t_curr in enumerate(times):
        hist_end = np.searchsorted(times, t_curr, side="left")
        if hist_end > 0:
            time_since_prev[idx[i]] = (t_curr - times[hist_end - 1]) / 60.0

        left_10m = np.searchsorted(times, t_curr - 600, side="left")
        left_1h = np.searchsorted(times, t_curr - 3600, side="left")
        left_24h = np.searchsorted(times, t_curr - 86400, side="left")
        left_30d = np.searchsorted(times, t_curr - 30 * 86400, side="left")

        c_10m[idx[i]] = hist_end - left_10m
        c_1h[idx[i]] = hist_end - left_1h
        c_24h[idx[i]] = hist_end - left_24h
        s_10m[idx[i]] = amounts[left_10m:hist_end].sum()
        s_1h[idx[i]] = amounts[left_1h:hist_end].sum()
        s_24h[idx[i]] = amounts[left_24h:hist_end].sum()
        if hist_end > left_30d:
            hist_median_30d[idx[i]] = np.median(amounts[left_30d:hist_end])

base_df["prior_txn_count_10m"] = c_10m
base_df["prior_txn_count_1h"] = c_1h
base_df["prior_txn_count_24h"] = c_24h
base_df["prior_txn_amount_sum_10m"] = s_10m
base_df["prior_txn_amount_sum_1h"] = s_1h
base_df["prior_txn_amount_sum_24h"] = s_24h
base_df["time_since_previous_txn_minutes"] = time_since_prev
base_df["historical_median_amount_30d"] = hist_median_30d

# 3. Tính toán các đặc trưng dẫn xuất
base_df["is_first_transaction"] = base_df["time_since_previous_txn_minutes"].isna().astype(int)
base_df["amount_to_historical_median_ratio"] = np.where(
    base_df["historical_median_amount_30d"].notna() & (base_df["historical_median_amount_30d"] > 0),
    base_df["amount_num"] / base_df["historical_median_amount_30d"],
    1.0,
)
base_df["is_extreme_median_spike"] = (base_df["amount_to_historical_median_ratio"] >= 5.0).astype(int)
base_df["avg_amount_per_recent_txn"] = np.where(
    base_df["prior_txn_count_24h"] > 0,
    base_df["prior_txn_amount_sum_24h"] / base_df["prior_txn_count_24h"],
    0.0,
)
base_df["velocity_intensity"] = (base_df["prior_txn_count_10m"] * base_df["amount_num"]) / (base_df["effective_limit"] + 1.0)
base_df["txn_count_10m_to_24h_ratio"] = (base_df["prior_txn_count_10m"] + 1.0) / (base_df["prior_txn_count_24h"] + 1.0)
base_df["amount_sum_1h_to_24h_ratio"] = (base_df["prior_txn_amount_sum_1h"] + 1.0) / (base_df["prior_txn_amount_sum_24h"] + 1.0)
base_df["velocity_x_median_spike"] = ((base_df["prior_txn_count_10m"] >= 2) & (base_df["amount_to_historical_median_ratio"] >= 3.0)).astype(int)

print(f"✓ Hoàn thành Mục 6: Velocity Features strictly < T trong {time.time() - t0:.2f}s.")

---
<a id="7"></a>
## 7. Đặc trưng Thời gian & Chu kỳ Tuần hoàn (Cyclical Temporal Features)

### Mục tiêu & Mã hóa Hàm Sóng ($\sin/\cos$)
- **Múi giờ**: Sử dụng thời gian chuẩn hóa tại Việt Nam (`Asia/Ho_Chi_Minh` — UTC+7).
- **Mã hóa Tuần hoàn**: Chuyển đổi 24 giờ trong ngày và 7 ngày trong tuần sang hàm sóng liên tục nhằm tránh bẫy shortcut của One-Hot Encoding:
  $$\text{hour\_sin} = \sin\left(\frac{2\pi \times \text{hour}}{24}\right), \quad \text{hour\_cos} = \cos\left(\frac{2\pi \times \text{hour}}{24}\right)$$
  $$\text{dow\_sin} = \sin\left(\frac{2\pi \times \text{dayofweek}}{7}\right), \quad \text{dow\_cos} = \cos\left(\frac{2\pi \times \text{dayofweek}}{7}\right)$$
- **Cờ Ngữ cảnh**: `is_night` ($00:00 - 05:59$) và `is_weekend` (Thứ 7 & Chủ Nhật).


In [ ]:
# 1. Trích xuất giờ và thứ địa phương (Asia/Ho_Chi_Minh)
hour_local = base_df["txn_time_local"].dt.hour
dow_local = base_df["txn_time_local"].dt.dayofweek  # 0: Monday, 6: Sunday

# 2. Mã hóa tuần hoàn chu kỳ 24 giờ và 7 ngày
base_df["hour_sin"] = np.sin(2 * np.pi * hour_local / 24.0)
base_df["hour_cos"] = np.cos(2 * np.pi * hour_local / 24.0)
base_df["dow_sin"] = np.sin(2 * np.pi * dow_local / 7.0)
base_df["dow_cos"] = np.cos(2 * np.pi * dow_local / 7.0)

# 3. Cờ ngữ cảnh thời gian
base_df["is_night"] = hour_local.between(0, 5).astype(int)
base_df["is_weekend"] = (dow_local >= 5).astype(int)

print("✓ Hoàn thành Mục 7: Mã hóa chu kỳ thời gian (hour_sin/cos, dow_sin/cos, is_night, is_weekend).")


---
<a id="8"></a>
## 8. Đặc trưng Thiết bị & Vị trí (Device & Geo-Location Dynamics)

### Mục tiêu & Tính toán Lịch sử Thiết bị
- `is_new_device`: thiết bị chưa từng xuất hiện với khách hàng tại timestamp **strictly trước** $T$; các giao dịch cùng timestamp dùng cùng snapshot lịch sử.
- `device_age_minutes`: khoảng cách từ `first_seen_at_utc` hợp lệ đến $T$. Timestamp `first_seen_at > T` bị cách ly thành missing, tuyệt đối không clip thành 0.
- `prior_device_txn_count`: số giao dịch của cặp `(customer_id, device_id)` có timestamp `< T`.
- Các tín hiệu mạng giữ nguyên từ session/device hiện tại; được đánh dấu shortcut risk trong registry.

In [ ]:
# 1. Audit timeline của device; timestamp tương lai không được đi vào feature.
base_df = base_df.sort_values(["txn_time_utc", "transaction_id"]).reset_index(drop=True)
base_df["device_id_missing"] = (base_df["device_id"].isna() | base_df["device_id"].eq("")).astype(int)
device_first_seen_future_mask = (
    base_df["first_seen_at_utc"].notna()
    & (base_df["first_seen_at_utc"] > base_df["txn_time_utc"])
)
valid_device_first_seen = base_df["first_seen_at_utc"].notna() & ~device_first_seen_future_mask
base_df["device_age_minutes"] = np.nan
base_df.loc[valid_device_first_seen, "device_age_minutes"] = (
    base_df.loc[valid_device_first_seen, "txn_time_utc"]
    - base_df.loc[valid_device_first_seen, "first_seen_at_utc"]
).dt.total_seconds() / 60.0

# 2. Cumcount tổng trừ cumcount trong cùng timestamp = số transaction strictly trước T.
valid_device = base_df["device_id"].notna() & base_df["device_id"].ne("")
device_view = base_df.loc[valid_device].sort_values(
    ["customer_id", "device_id", "txn_time_utc", "transaction_id"]
)
device_position = device_view.groupby(["customer_id", "device_id"], sort=False).cumcount()
same_time_position = device_view.groupby(
    ["customer_id", "device_id", "txn_time_utc"], sort=False
).cumcount()
prior_device = (device_position - same_time_position).astype(np.int32)
base_df["prior_device_txn_count"] = 0
base_df.loc[device_view.index, "prior_device_txn_count"] = prior_device.to_numpy()
base_df["is_new_device"] = 0
base_df.loc[device_view.index, "is_new_device"] = prior_device.eq(0).astype(np.int32).to_numpy()

print("✓ Hoàn thành Mục 8: Đặc trưng Thiết bị & Vị trí")
print(f"  - Device first_seen ở tương lai đã cách ly: {device_first_seen_future_mask.sum():,}")
print(f"  - Giao dịch trên thiết bị mới: {base_df['is_new_device'].sum():,} ({base_df['is_new_device'].mean():.2%})")

---
<a id="9"></a>
## 9. Đặc trưng Người Thụ Hưởng (Beneficiary Dynamics)

### Mục tiêu & Tính toán Lịch sử Người Nhận
- `is_external_transfer`: Cờ chuyển tiền ngoài ngân hàng (`is_internal_bank == False` hoặc ngân hàng thụ hưởng khác ngân hàng nguồn).
- `time_since_beneficiary_added_minutes`: Khoảng thời gian từ khi thêm người thụ hưởng (`bene_added_at_utc`) đến thời điểm giao dịch $T$. Đảm bảo điều kiện $\text{added\_at} \le T$.
- `is_new_beneficiary`: Người thụ hưởng mới thêm trong vòng 24 giờ ($\le 1440$ phút).
- `prior_beneficiary_txn_count`: Số lần tài khoản nguồn đã từng chuyển tới `beneficiary_id` này trước $T$.


In [ ]:
if "is_internal_bank" not in base_df.columns:
    bene_map = df_bens.set_index("beneficiary_id")["is_internal_bank"].to_dict()
    base_df["is_internal_bank"] = base_df["beneficiary_id"].map(bene_map)

# 1. added_at phải xảy ra trước hoặc tại T; dữ liệu tương lai bị giữ missing thay vì clip thành 0.
bene_time_valid = base_df["bene_added_at_utc"].notna() & (base_df["bene_added_at_utc"] <= base_df["txn_time_utc"])
base_df["time_since_beneficiary_added_minutes"] = np.nan
base_df.loc[bene_time_valid, "time_since_beneficiary_added_minutes"] = (
    base_df.loc[bene_time_valid, "txn_time_utc"] - base_df.loc[bene_time_valid, "bene_added_at_utc"]
).dt.total_seconds() / 60.0
base_df["is_new_beneficiary"] = (
    base_df["time_since_beneficiary_added_minutes"].notna()
    & base_df["time_since_beneficiary_added_minutes"].le(1440)
).astype(int)
base_df["is_external_transfer"] = base_df["is_internal_bank"].astype(str).str.lower().isin(["false", "0", "0.0"]).astype(int)

# 2. Loại toàn bộ peer cùng T khỏi lịch sử account-beneficiary.
valid_bene = base_df["beneficiary_id"].notna() & base_df["beneficiary_id"].ne("")
bene_view = base_df.loc[valid_bene].sort_values(
    ["account_id", "beneficiary_id", "txn_time_utc", "transaction_id"]
)
bene_position = bene_view.groupby(["account_id", "beneficiary_id"], sort=False).cumcount()
same_time_position = bene_view.groupby(
    ["account_id", "beneficiary_id", "txn_time_utc"], sort=False
).cumcount()
prior_bene = (bene_position - same_time_position).astype(np.int32)
base_df["prior_beneficiary_txn_count"] = 0
base_df.loc[bene_view.index, "prior_beneficiary_txn_count"] = prior_bene.to_numpy()

print("✓ Hoàn thành Mục 9: Beneficiary features strictly < T")
print(f"  - Giao dịch cho beneficiary mới: {base_df['is_new_beneficiary'].sum():,}")

---
<a id="10"></a>
## 10. Đặc trưng Xác thực & Biến động Nhạy cảm (Auth & Account Changes)

### Mục tiêu & Tính toán Point-in-Time Sự kiện Bảo mật
- **Sự kiện Xác thực (Auth Events strictly $< T$)**:
  - `failed_auth_count_30m`: Số lần xác thực thất bại (`auth_result != SUCCESS`) trong 30 phút trước $T$.
  - `failed_auth_count_24h`: Số lần xác thực thất bại trong 24 giờ trước $T$.
  - `time_since_last_failed_auth_minutes`: Khoảng thời gian từ lần thất bại xác thực gần nhất đến $T$.
- **Sự kiện Thay đổi Nhạy cảm (Sensitive Change Events strictly $< T$)**:
  - Đổi mật khẩu, đổi SĐT, đổi thiết bị, reset phương thức bảo mật.
  - `time_since_sensitive_change_minutes`: Thời gian từ lần đổi thông tin gần nhất đến $T$.
  - `has_sensitive_change_history`: Tài khoản đã từng có lịch sử thay đổi thông tin hay chưa.
  - `is_after_sensitive_change`: Giao dịch phát sinh trong vòng 60 phút sau khi đổi thông tin.


In [ ]:
# 1. Chuẩn bị bảng auth_events và account_change_events
df_auths["auth_time_int"] = df_auths["auth_at_utc"].astype(np.int64) // 10**9
failed_auths = df_auths[df_auths["auth_result"].str.lower() != "success"].sort_values(["account_id", "auth_time_int"])

df_changes["change_time_int"] = df_changes["changed_at_utc"].astype(np.int64) // 10**9
changes = df_changes.sort_values(["account_id", "change_time_int"])

# 2. Vector hóa tính toán strictly < T
failed_30m = np.zeros(len(base_df), dtype=np.int32)
failed_24h = np.zeros(len(base_df), dtype=np.int32)
time_since_last_fail = np.full(len(base_df), np.nan, dtype=np.float64)
time_since_change = np.full(len(base_df), np.nan, dtype=np.float64)

auth_groups = failed_auths.groupby("account_id")
change_groups = changes.groupby("account_id")

t0 = time.time()
for acc_id, group_indices in base_df.groupby("account_id").groups.items():
    idx = group_indices.values
    t_txns = base_df.loc[idx, "txn_time_int"].values
    
    # Khớp failed auths
    if acc_id in auth_groups.groups:
        auth_times = failed_auths.loc[auth_groups.groups[acc_id], "auth_time_int"].values
        for i in range(len(t_txns)):
            t_curr = t_txns[i]
            right = np.searchsorted(auth_times, t_curr, side="left")
            if right > 0:
                left_30m = np.searchsorted(auth_times[:right], t_curr - 1800, side="left")
                left_24h = np.searchsorted(auth_times[:right], t_curr - 86400, side="left")
                failed_30m[idx[i]] = right - left_30m
                failed_24h[idx[i]] = right - left_24h
                time_since_last_fail[idx[i]] = (t_curr - auth_times[right - 1]) / 60.0

    # Khớp account changes
    if acc_id in change_groups.groups:
        ch_times = changes.loc[change_groups.groups[acc_id], "change_time_int"].values
        for i in range(len(t_txns)):
            t_curr = t_txns[i]
            right = np.searchsorted(ch_times, t_curr, side="left")
            if right > 0:
                time_since_change[idx[i]] = (t_curr - ch_times[right - 1]) / 60.0

base_df["failed_auth_count_30m"] = failed_30m
base_df["failed_auth_count_24h"] = failed_24h
base_df["time_since_last_failed_auth_minutes"] = time_since_last_fail
base_df["time_since_sensitive_change_minutes"] = time_since_change

# 3. Cờ dẫn xuất
base_df["has_sensitive_change_history"] = base_df["time_since_sensitive_change_minutes"].notna().astype(int)
base_df["is_after_sensitive_change"] = (base_df["time_since_sensitive_change_minutes"].notna() & 
                                        (base_df["time_since_sensitive_change_minutes"] <= 60.0)).astype(int)

print(f"✓ Hoàn thành Mục 10: Auth & Sensitive Changes trong {time.time() - t0:.2f}s:")
print(f"  - Số GD phát sinh sau khi đổi TT nhạy cảm <= 60m : {base_df['is_after_sensitive_change'].sum():,}")
print(f"  - Số GD có thất bại xác thực trong 30m          : {(base_df['failed_auth_count_30m'] > 0).sum():,}")


---
<a id="11"></a>
## 11. Đặc trưng Chuỗi Tương tác Phức hợp (Composite Attack Sequences)

### Mục tiêu & Các Tổ Hợp Chuỗi Tấn Công
- **Mục tiêu**: Kết hợp các tín hiệu đơn lẻ đã được kiểm chứng bằng Contingency Table ở EDA thành các đặc trưng chuỗi có độ chính xác cực cao ($> 90\%$).
- **Danh mục Đặc trưng Chuỗi**:
  1. `is_ato_sequence`: $\text{is\_after\_sensitive\_change} \land \text{is\_new\_device}$ (Lift = 46.57x).
  2. `is_rapid_new_beneficiary`: $\text{is\_new\_beneficiary} \land (\text{time\_since\_beneficiary\_added\_minutes} \le 15)$ (Lift = 46.57x).
  3. `is_ato_fast_drain`: $\text{is\_after\_sensitive\_change} \land (\text{time\_since\_sensitive\_change} \le 30\text{m}) \land \text{is\_high\_balance\_drain}$.
  4. `is_bot_network`: $\text{is\_emulator\_int} \land \text{proxy\_flag\_int}$.
  5. `new_device_x_high_amount`: $\text{is\_new\_device} \land \text{is\_high\_limit\_usage}$.
  6. `proxy_x_new_device`: $\text{proxy\_flag\_int} \land \text{is\_new\_device}$.
- **Gắn nhãn Rủi ro Shortcut (Shortcut Risk Tag)**: Các biến có tỷ lệ lift quá lớn hoặc gắn liền với kịch bản template được đánh dấu `shortcut_risk = HIGH` phục vụ bài toán Ablation Study ở Notebook 05.


In [ ]:
# 1. Xây dựng các đặc trưng chuỗi phức hợp
base_df["is_ato_sequence"] = (
    (base_df["is_after_sensitive_change"] == 1) & 
    (base_df["is_new_device"] == 1)
).astype(int)

base_df["is_rapid_new_beneficiary"] = (
    (base_df["is_new_beneficiary"] == 1) & 
    (base_df["time_since_beneficiary_added_minutes"].notna()) & 
    (base_df["time_since_beneficiary_added_minutes"] <= 15.0)
).astype(int)

base_df["is_ato_fast_drain"] = (
    (base_df["is_after_sensitive_change"] == 1) & 
    (base_df["time_since_sensitive_change_minutes"].notna()) & 
    (base_df["time_since_sensitive_change_minutes"] <= 30.0) & 
    (base_df["is_high_balance_drain"] == 1)
).astype(int)

base_df["is_bot_network"] = (
    (base_df["is_emulator_int"] == 1) & 
    (base_df["proxy_flag_int"] == 1)
).astype(int)

base_df["new_device_x_high_amount"] = (
    (base_df["is_new_device"] == 1) & 
    (base_df["is_high_limit_usage"] == 1)
).astype(int)

base_df["proxy_x_new_device"] = (
    (base_df["proxy_flag_int"] == 1) & 
    (base_df["is_new_device"] == 1)
).astype(int)

print("✓ Hoàn thành Mục 11: Đặc trưng Chuỗi Tương tác Phức hợp")
print(f"  - Chuỗi ATO (is_ato_sequence)                : {base_df['is_ato_sequence'].sum():,} GD")
print(f"  - Chuỗi Scam Gấp (is_rapid_new_beneficiary)  : {base_df['is_rapid_new_beneficiary'].sum():,} GD")
print(f"  - Chuỗi Bot Farm (is_bot_network)            : {base_df['is_bot_network'].sum():,} GD")


---
<a id="12"></a>
## 12. Chính sách Missing & Biến đổi Duration

Các duration `time_since_*` giữ bản raw để tính rule/composite. Đầu vào model dùng phiên bản `_log1p`: missing “không có lịch sử” được mã hóa bằng sentinel trước khi `log1p`, giúp Logistic không bị trị tuyệt đối `999999` chi phối. `device_age_minutes` tiếp tục để NaN và được median-impute chỉ bên trong pipeline train.

Boolean context thiếu sau join được ghi nhận trong `context_missing_counts` rồi điền `0`; categorical context dùng `UNKNOWN`.

In [ ]:
TIME_SINCE_RAW_FEATURES = [
    "time_since_previous_txn_minutes",
    "time_since_beneficiary_added_minutes",
    "time_since_sensitive_change_minutes",
    "time_since_last_failed_auth_minutes",
]
TIME_SINCE_LOG_FEATURES = [f"{col}_log1p" for col in TIME_SINCE_RAW_FEATURES]
NO_HISTORY_SENTINEL_MINUTES = 999999.0

# 1. Giữ raw duration cho audit/rule; model sử dụng log1p duration.
for raw_col, log_col in zip(TIME_SINCE_RAW_FEATURES, TIME_SINCE_LOG_FEATURES):
    filled = base_df[raw_col].fillna(NO_HISTORY_SENTINEL_MINUTES)
    assert filled.ge(0).all(), f"{raw_col} chứa duration âm"
    base_df[log_col] = np.log1p(filled)

# 2. Missing context được ghi nhận trước khi áp dụng policy rõ ràng.
binary_context_cols = [
    "is_new_location_int", "proxy_flag_int", "vpn_flag_int",
    "is_emulator_int", "is_rooted_int",
]
context_missing_counts = {col: int(base_df[col].isna().sum()) for col in binary_context_cols}
for col in binary_context_cols:
    base_df[col] = base_df[col].fillna(0).astype(np.int8)

cat_impute_cols = [
    "channel", "customer_segment", "kyc_level", "base_risk_level", "occupation_group",
    "device_type", "os", "trust_status", "account_type", "province", "province_x", "province_y",
]
for col in cat_impute_cols:
    if col in base_df.columns:
        base_df[col] = base_df[col].fillna("UNKNOWN")

numeric_check_cols = [
    "log_amount", "amount_to_limit_ratio", "amount_to_pre_txn_balance_ratio", "amount_to_historical_median_ratio",
    "prior_txn_count_10m", "prior_txn_count_1h", "prior_txn_count_24h",
    "prior_txn_amount_sum_10m", "prior_txn_amount_sum_1h", "prior_txn_amount_sum_24h",
    "velocity_intensity", *TIME_SINCE_LOG_FEATURES,
]
for col in numeric_check_cols:
    assert np.isfinite(base_df[col]).all(), f"{col} chứa NaN/inf ngoài policy"
assert not np.isinf(base_df["device_age_minutes"].dropna()).any()

print("✓ Missing/context policy và log1p duration đã áp dụng.")
print("  - Missing context trước fill:", context_missing_counts)

---
<a id="13"></a>
## 13. Kiểm toán Rò rỉ Dữ liệu & Danh mục Cấm (Leakage & Deny-List Audit)

### Mục tiêu & Danh mục Loại bỏ Tuyệt đối (Deny-List)
- **Mục tiêu**: Rà soát và loại bỏ toàn bộ các cột, chuỗi ký tự hoặc thông tin hậu nghiệm có thể gây rò rỉ nhãn (Data Leakage) hoặc tạo lối tắt nhân tạo (Synthetic Shortcut).
- **Danh mục Cấm**:
  - `scenario_code`, `scenario_hint`, `mule_cluster_id`, `event_id`, `fraud_label`, `investigation_outcome`, `population`, `entity_role`.
  - Bất kỳ chuỗi định danh nào chứa tiền tố `_SCN_`, `RING_`, `FRAUD_`.
  - Các biến định danh nhạy cảm: `customer_id`, `account_id`, `session_id`, `device_id`, `beneficiary_id` (được đưa vào bảng `AUDIT` riêng, không nằm trong `X`).


In [ ]:
# 1. Danh sách tất cả các cột đặc trưng mô hình được phép sử dụng (Allow-List sạch 100%)
MODEL_FEATURES = [
    # Nhóm 1: Tài chính & Hạn mức
    "log_amount", "amount_to_limit_ratio", "amount_to_pre_txn_balance_ratio", "amount_to_historical_median_ratio",
    "is_high_limit_usage", "is_high_balance_drain", "is_extreme_median_spike",
    
    # Nhóm 2: Velocity & Lịch sử
    "prior_txn_count_10m", "prior_txn_count_1h", "prior_txn_count_24h",
    "prior_txn_amount_sum_10m", "prior_txn_amount_sum_1h", "prior_txn_amount_sum_24h",
    "time_since_previous_txn_minutes_log1p", "avg_amount_per_recent_txn", "velocity_intensity",
    "txn_count_10m_to_24h_ratio", "amount_sum_1h_to_24h_ratio", "velocity_x_median_spike", "is_first_transaction",
    
    # Nhóm 3: Chu kỳ thời gian
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_night", "is_weekend",
    
    # Nhóm 4: Thiết bị & Vị trí (Đã loại bỏ device_id_missing do zero-variance)
    "is_new_device", "device_age_minutes", "prior_device_txn_count",
    "is_new_location_int", "proxy_flag_int", "vpn_flag_int", "is_emulator_int", "is_rooted_int",
    
    # Nhóm 5: Người thụ hưởng
    "is_external_transfer", "is_new_beneficiary", "time_since_beneficiary_added_minutes_log1p", "prior_beneficiary_txn_count",
    
    # Nhóm 6: Xác thực & Đổi TT nhạy cảm
    "failed_auth_count_30m", "failed_auth_count_24h", "time_since_last_failed_auth_minutes_log1p",
    "time_since_sensitive_change_minutes_log1p", "is_after_sensitive_change",
    
    # Nhóm 7: Chuỗi tương tác phức hợp
    "is_ato_sequence", "is_rapid_new_beneficiary", "is_ato_fast_drain",
    "new_device_x_high_amount",
    
    # Nhóm 8: Categorical
    "channel", "customer_segment", "kyc_level", "base_risk_level", "account_type", "device_type", "os"
]

# 2. Assertions kiểm toán Deny-List
for deny_col in DENY_COLS:
    assert deny_col not in MODEL_FEATURES, f"LỖI LEAKAGE NGHIÊM TRỌNG: Cột cấm '{deny_col}' xuất hiện trong MODEL_FEATURES!"

# 3. Quét kiểm tra dấu vết _SCN_ hoặc RING_ trong dữ liệu MODEL_FEATURES
for col in MODEL_FEATURES:
    if base_df[col].dtype == object:
        leakage_mask = base_df[col].astype(str).str.contains(r"_SCN_|RING_|SCENARIO", regex=True)
        assert not leakage_mask.any(), f"LỖI LEAKAGE: Phát hiện chuỗi kịch bản generator trong cột '{col}'!"

print(f"✓ Kiểm toán Leakage hoàn tất 100%: Toàn bộ {len(MODEL_FEATURES)} đặc trưng trong MODEL_FEATURES đạt chuẩn Anti-Leakage.")

---
<a id="14"></a>
## 14. Kiểm tra Toàn diện Feature Matrix (Matrix Validation Assertions)

### Mục tiêu & 10 Tiêu Chí Kiểm Định
- **10 Tiêu chí Kiểm định Bắt buộc**:
  1. Số dòng bảo toàn chính xác bằng `transactions.csv` ($111.064$ dòng).
  2. `transaction_id` duy nhất ($0$ duplicate).
  3. $0$ giá trị `NaN` hoặc `inf` ngoài chính sách missing.
  4. $0$ cột trong danh mục cấm (`DENY_COLS`).
  5. Cửa sổ rolling velocity loại trừ giao dịch hiện tại ($T$).
  6. $0$ sự kiện bảo mật/thay đổi xảy ra trong tương lai ($> T$).
  7. Toàn bộ đặc trưng có kiểu dữ liệu hợp lệ (`float64`, `int32`, `object`).
  8. Phân phối ổn định trên từng `simulation_run_id`.
  9. Tỷ lệ nhãn mục tiêu (`target_fraud`) được bảo toàn trên toàn tập.
  10. $0$ đặc trưng có phương sai bằng 0 (Zero Variance Feature Check).


In [ ]:
# Hard gate dừng notebook khi artifact có thể sai; soft gate chỉ cảnh báo profile thay đổi.
if "simulation_run_id" not in base_df.columns:
    if "simulation_run_id_x" in base_df.columns:
        base_df["simulation_run_id"] = base_df["simulation_run_id_x"]
    elif "simulation_run_id_y" in base_df.columns:
        base_df["simulation_run_id"] = base_df["simulation_run_id_y"]

validation_report = []
def add_check(name, passed, detail, severity="HARD"):
    validation_report.append({
        "Tiêu chí (Validation Criteria)": name,
        "Severity": severity,
        "Kết quả": "PASS" if passed else ("FAIL" if severity == "HARD" else "WARN"),
        "Chi tiết": detail,
    })

add_check("1. Row count bảo toàn so với manifest", len(base_df) == expected_txn_count, f"{len(base_df):,} dòng")
add_check("2. Tính duy nhất của transaction_id", base_df["transaction_id"].is_unique, f"{base_df['transaction_id'].duplicated().sum()} duplicates")

allowed_missing_features = {"device_age_minutes"}
unexpected_missing = int(base_df[[c for c in MODEL_FEATURES if c not in allowed_missing_features]].isna().sum().sum())
add_check("3. Không có NaN ngoài chính sách missing", unexpected_missing == 0, f"{unexpected_missing} NaN ngoài chính sách")

deny_found = [c for c in DENY_COLS if c in MODEL_FEATURES]
add_check("4. Không chứa cột trong Deny-List", not deny_found, f"{len(deny_found)} deny cols")

# Peer-snapshot audit: mọi dòng cùng entity + T phải thấy cùng lịch sử strictly < T.
account_snapshot = base_df.groupby(["account_id", "txn_time_utc"], sort=False).agg(
    rows=("transaction_id", "size"), count_values=("prior_txn_count_24h", "nunique"),
    amount_values=("prior_txn_amount_sum_24h", "nunique"),
)
account_peer_violations = int(((account_snapshot["rows"] > 1) & ((account_snapshot["count_values"] > 1) | (account_snapshot["amount_values"] > 1))).sum())
first_account_time = base_df.groupby("account_id")["txn_time_utc"].transform("min")
first_account_violations = int((base_df.loc[base_df["txn_time_utc"].eq(first_account_time), "prior_txn_count_24h"] != 0).sum())

device_rows = base_df.loc[base_df["device_id"].notna() & base_df["device_id"].ne("")]
device_snapshot = device_rows.groupby(["customer_id", "device_id", "txn_time_utc"], sort=False).agg(
    rows=("transaction_id", "size"), prior_values=("prior_device_txn_count", "nunique"),
)
device_peer_violations = int(((device_snapshot["rows"] > 1) & (device_snapshot["prior_values"] > 1)).sum())
first_device_time = device_rows.groupby(["customer_id", "device_id"])["txn_time_utc"].transform("min")
first_device_violations = int((device_rows.loc[device_rows["txn_time_utc"].eq(first_device_time), "prior_device_txn_count"] != 0).sum())

bene_rows = base_df.loc[base_df["beneficiary_id"].notna() & base_df["beneficiary_id"].ne("")]
bene_snapshot = bene_rows.groupby(["account_id", "beneficiary_id", "txn_time_utc"], sort=False).agg(
    rows=("transaction_id", "size"), prior_values=("prior_beneficiary_txn_count", "nunique"),
)
bene_peer_violations = int(((bene_snapshot["rows"] > 1) & (bene_snapshot["prior_values"] > 1)).sum())
first_bene_time = bene_rows.groupby(["account_id", "beneficiary_id"])["txn_time_utc"].transform("min")
first_bene_violations = int((bene_rows.loc[bene_rows["txn_time_utc"].eq(first_bene_time), "prior_beneficiary_txn_count"] != 0).sum())

pit_violations = sum([account_peer_violations, first_account_violations, device_peer_violations, first_device_violations, bene_peer_violations, first_bene_violations])
add_check(
    "5. Rolling causality strictly < T", pit_violations == 0,
    f"account_peer={account_peer_violations}, device_peer={device_peer_violations}, bene_peer={bene_peer_violations}; first-history={first_account_violations + first_device_violations + first_bene_violations}",
)

future_rows = int(device_first_seen_future_mask.sum())
future_reached_feature = int(base_df.loc[device_first_seen_future_mask, "device_age_minutes"].notna().sum())
valid_age_negative = int((base_df.loc[~device_first_seen_future_mask, "device_age_minutes"].dropna() < 0).sum())
add_check(
    "6. Future device first_seen không đi vào feature",
    future_reached_feature == 0 and valid_age_negative == 0,
    f"{future_rows} source rows cách ly; {future_reached_feature} future rows lọt vào feature",
)

allowed_types = [np.dtype("float64"), np.dtype("float32"), np.dtype("int64"), np.dtype("int32"), np.dtype("int16"), np.dtype("int8"), np.dtype("bool"), np.dtype("O")]
invalid_dtypes = [col for col in MODEL_FEATURES if base_df[col].dtype not in allowed_types]
add_check("7. Kiểu dữ liệu hợp lệ", not invalid_dtypes, f"{len(invalid_dtypes)} cột sai: {invalid_dtypes}")

fraud_rate = base_df["target_fraud"].mean() * 100
add_check("8. Fraud-rate nằm trong profile tham chiếu", 2.0 <= fraud_rate <= 2.3, f"{fraud_rate:.2f}% (reference 2.0%–2.3%)", "SOFT")

zero_var_cols = [c for c in MODEL_FEATURES if base_df[c].dtype != object and base_df[c].nunique(dropna=False) <= 1]
add_check("9. Không có feature phương sai bằng 0", not zero_var_cols, f"{len(zero_var_cols)} cột: {zero_var_cols}", "SOFT")

manifest_run_count = int(manifest["run_count"])
n_runs = base_df["simulation_run_id"].nunique()
add_check("10. Run coverage khớp manifest", n_runs == manifest_run_count and n_runs >= 3, f"actual={n_runs}, manifest={manifest_run_count}")

val_df = pd.DataFrame(validation_report)
print("=== BÁO CÁO KIỂM TOÁN FEATURE MATRIX ===")
display(val_df)
hard_failures = val_df.loc[(val_df["Severity"] == "HARD") & (val_df["Kết quả"] == "FAIL")]
assert hard_failures.empty, f"Hard gate thất bại: {hard_failures['Tiêu chí (Validation Criteria)'].tolist()}"
if val_df["Kết quả"].eq("WARN").any():
    print("⚠ Soft warnings không chặn research/demo; phải ghi nhận khi diễn giải kết quả.")

---
<a id="15"></a>
## 15. Phân chia Tập Dữ liệu An toàn (Entity-Safe & Temporal Split)

- Đọc thứ tự `simulation_run_id` từ manifest: các run trừ hai run cuối dùng cho train, run áp chót dùng validation, run cuối là test holdout.
- Cần tối thiểu ba run duy nhất; không hard-code số dòng hoặc ID run trong narrative.
- Account và customer intersection giữa mọi cặp train/validation/test bắt buộc bằng rỗng.
- Notebook 03 chỉ xuất raw feature splits; preprocessing được fit bên trong từng pipeline/fold ở Notebook 04.

In [ ]:
# 1. Audit columns và split run được lấy từ manifest, không hard-code seed/run ID trong logic.
if "event_id" not in base_df.columns:
    bridge_txn = raw_bridge[raw_bridge["entity_type"] == "transaction"].drop_duplicates(subset="entity_id").set_index("entity_id")
    base_df["event_id"] = base_df["transaction_id"].map(bridge_txn["event_id"]).fillna("none")
    base_df["scenario_code"] = base_df["transaction_id"].map(bridge_txn["scenario_code"]).fillna("BACKGROUND")
    base_df["label_scope"] = base_df["transaction_id"].map(bridge_txn["label_scope"]).fillna("background")

AUDIT_COLS = [
    "transaction_id", "account_id", "customer_id", "simulation_run_id",
    "event_id", "scenario_code", "label_scope", "sample_weight", "hard_negative",
    "amount_num", "beneficiary_id", "txn_time_utc",
]
AUDIT_COLS = [c for c in AUDIT_COLS if c in base_df.columns]
run_ids = [item["simulation_run_id"] for item in manifest["runs"]]
assert len(run_ids) >= 3 and len(run_ids) == len(set(run_ids)), "Cần ít nhất 3 run duy nhất để train/validation/test"
train_run_ids, validation_run_id, test_run_id = run_ids[:-2], run_ids[-2], run_ids[-1]

train_mask = base_df["simulation_run_id"].isin(train_run_ids)
valid_mask = base_df["simulation_run_id"].eq(validation_run_id)
test_mask = base_df["simulation_run_id"].eq(test_run_id)
df_train = base_df.loc[train_mask].copy().reset_index(drop=True)
df_valid = base_df.loc[valid_mask].copy().reset_index(drop=True)
df_test = base_df.loc[test_mask].copy().reset_index(drop=True)
assert min(len(df_train), len(df_valid), len(df_test)) > 0, "Một split không có dữ liệu"

# 2. Entity-safe isolation là hard gate, không được nới vì demo.
for left_name, left_df, right_name, right_df in [
    ("train", df_train, "validation", df_valid),
    ("train", df_train, "test", df_test),
    ("validation", df_valid, "test", df_test),
]:
    for entity in ["account_id", "customer_id"]:
        overlap = set(left_df[entity]).intersection(right_df[entity])
        assert not overlap, f"LEAKAGE: {len(overlap)} {entity} trùng giữa {left_name}/{right_name}"

X_train, y_train = df_train[MODEL_FEATURES], df_train[TARGET_COL]
audit_train = df_train[AUDIT_COLS]

print("=== ENTITY-SAFE & MANIFEST-DRIVEN SPLIT ===")
print(f"  - Train {train_run_ids}: {len(df_train):,} | Fraud {y_train.mean():.2%}")
print(f"  - Validation [{validation_run_id}]: {len(df_valid):,}")
print(f"  - Test [{test_run_id}]: {len(df_test):,} — chỉ xuất holdout, không transform tại Notebook 03")

---
### 15.2 Phân tích Tương quan & Trích chọn Đặc trưng trên Tập Train (Feature Selection)

### Mục tiêu
- **Tính toán ma trận tương quan Pearson ($r$)** giữa các biến số học trên **chính tập Train (`X_train`)** để nhận diện hiện tượng đa cộng tuyến nghiêm trọng ($|r| \ge 0.85$).
- **Đánh giá lực phân tách (Separability / Correlation with Target)** đối với nhãn gian lận `y_train`.
- **Lựa chọn danh mục đặc trưng tối ưu (`SELECTED_FEATURES`)** để chuyển sang Pipeline mô hình hóa.


In [ ]:
from sklearn.feature_selection import mutual_info_classif

# 1. Trích xuất các biến số học trên X_train
numeric_cols_for_analysis = [
    c for c in MODEL_FEATURES 
    if X_train[c].dtype in [np.float64, np.float32, np.int64, np.int32, np.int8]
]

# 2. Tính toán Ma trận Tương quan Pearson trên X_train
corr_matrix = X_train[numeric_cols_for_analysis].corr()

# ─────────────────────────────────────────────────────────────────────────────
# PHẦN 1: TRỰC QUAN HÓA MA TRẬN TƯƠNG QUAN (CORRELATION HEATMAP CHO SLIDE/BÁO CÁO)
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(220, 20, as_cmap=True)

sns.heatmap(
    corr_matrix, mask=mask, cmap=cmap, vmax=1.0, vmin=-0.3, center=0,
    annot=False, square=True, linewidths=0.3,
    cbar_kws={"shrink": 0.75, "label": "Hệ số tương quan Pearson (r)"}, ax=ax
)
clean_ax(
    ax, "Ma Trận Tương Quan Đa Biến Trên Tập Train (Train Correlation Matrix)", 
    subtitle=f"Rà soát tương quan đa biến và đa cộng tuyến trên X_train ({len(X_train):,} GD)"
)
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# PHẦN 2: BẢNG SO SÁNH 3 THƯỚC ĐO: PEARSON + SPEARMAN + MUTUAL INFORMATION
# ─────────────────────────────────────────────────────────────────────────────
print("Đang tính toán Mutual Information & Spearman với y_train...")
t0 = time.time()

pearson_corr = X_train[numeric_cols_for_analysis].apply(lambda c: c.corr(y_train))
spearman_corr = X_train[numeric_cols_for_analysis].apply(lambda c: c.corr(y_train, method="spearman"))
mi_scores = pd.Series(
    mutual_info_classif(X_train[numeric_cols_for_analysis].fillna(0), y_train, random_state=42),
    index=numeric_cols_for_analysis
)

feature_importance_analysis = pd.DataFrame({
    "Pearson (r)": pearson_corr,
    "Spearman (rho)": spearman_corr,
    "Mutual Information (MI)": mi_scores
}).sort_values(by="Mutual Information (MI)", ascending=False)

print(f"✓ Hoàn thành phân tích lực phân tách trong {time.time() - t0:.2f}s:")
print("=== TOP 15 ĐẶC TRƯNG CÓ MUTUAL INFORMATION & TƯƠNG QUAN CAO NHẤT VỚI TARGET_FRAUD ===")
display(feature_importance_analysis.head(15))

# ─────────────────────────────────────────────────────────────────────────────
# PHẦN 3: BẢNG RÀ SOÁT CÁC CẶP BIẾN ĐA CỘNG TUYẾN (|r| >= 0.85)
# ─────────────────────────────────────────────────────────────────────────────
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        col1 = corr_matrix.columns[i]
        col2 = corr_matrix.columns[j]
        r_val = corr_matrix.iloc[i, j]
        if abs(r_val) >= 0.85:
            high_corr_pairs.append({
                "Biến 1": col1, 
                "Biến 2": col2, 
                "Hệ số r": f"{r_val:.3f}", 
                "Tương quan Target (Biến 1)": f"{pearson_corr.get(col1, 0):.3f}",
                "Tương quan Target (Biến 2)": f"{pearson_corr.get(col2, 0):.3f}"
            })

print("\n=== DANH SÁCH CÁC CẶP BIẾN ĐA CỘNG TUYẾN TRÊN TẬP TRAIN (|r| >= 0.85) ===")
display(pd.DataFrame(high_corr_pairs))

---
### 15.4 Thực Hiện Trích Chọn Đặc Trưng Tinh Gọn (Feature Pruning on Train)

- **Mục tiêu**: Loại bỏ các đặc trưng đa cộng tuyến ($|r| \ge 0.85$), giữ lại đặc trưng có lực phân tách tốt nhất với `target_fraud`.
- **Đầu ra**: Tập đặc trưng tinh gọn `SELECTED_FEATURES` sẵn sàng cho pipeline tiền xử lý và huấn luyện mô hình.


In [ ]:
# 1. Các cột dưới đây được tính cho audit/storytelling nhưng không đưa vào model do trùng tuyệt đối.
IDENTICAL_DUPLICATES = ["proxy_x_new_device", "is_bot_network", "has_sensitive_change_history"]
FULL_FEATURES = [c for c in MODEL_FEATURES if c not in IDENTICAL_DUPLICATES]

# 2. Shortcut tier chỉ liệt kê feature còn thực sự tồn tại trong FULL_FEATURES.
SHORTCUT_CANDIDATES = [
    "proxy_flag_int", "vpn_flag_int", "is_emulator_int", "is_rooted_int",
    "is_ato_sequence", "is_rapid_new_beneficiary", "is_ato_fast_drain", "new_device_x_high_amount",
]
NO_SHORTCUT_FEATURES = [c for c in FULL_FEATURES if c not in SHORTCUT_CANDIDATES]

CORE_FEATURES_BASE = [
    "log_amount", "amount_to_limit_ratio", "amount_to_pre_txn_balance_ratio", "amount_to_historical_median_ratio",
    "is_high_limit_usage", "is_high_balance_drain", "is_extreme_median_spike",
    "prior_txn_count_10m", "prior_txn_count_1h", "prior_txn_count_24h",
    "prior_txn_amount_sum_10m", "prior_txn_amount_sum_1h", "prior_txn_amount_sum_24h",
    "time_since_previous_txn_minutes_log1p", "avg_amount_per_recent_txn", "velocity_intensity", "is_first_transaction",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_night", "is_weekend",
    "is_new_device", "device_age_minutes", "prior_device_txn_count",
    "is_external_transfer", "is_new_beneficiary", "prior_beneficiary_txn_count",
    "channel", "customer_segment", "kyc_level", "base_risk_level", "account_type",
]
CORE_FEATURES = [c for c in CORE_FEATURES_BASE if c in FULL_FEATURES]

print("=== BA FEATURE TIERS CHO MODEL SELECTION ===")
print(f"  FULL        : {len(FULL_FEATURES)}")
print(f"  NO_SHORTCUT : {len(NO_SHORTCUT_FEATURES)}")
print(f"  CORE        : {len(CORE_FEATURES)}")
print("✓ Preprocessing chỉ tồn tại bên trong pipeline của Notebook 04.")

---
<a id="16"></a>
## 16. Bảng Đăng ký Hợp đồng Đặc trưng (Feature Contract Registry)

Registry phải phủ đủ toàn bộ FULL tier, gồm công thức `_log1p`, source, point-in-time rule, missing policy và SAS mapping. Assertion trong cell kế tiếp ngăn thiếu feature âm thầm.

In [ ]:
# Metadata được khai báo cho toàn bộ FULL_FEATURES; assertion cuối cell ngăn registry thiếu âm thầm.
FORMULAS = {
    "log_amount": "log1p(amount)",
    "amount_to_limit_ratio": "amount / min(single_txn_limit, daily_transfer_limit)",
    "amount_to_pre_txn_balance_ratio": "amount / balance_before",
    "amount_to_historical_median_ratio": "amount / median(prior amount, 30d)",
    "is_high_limit_usage": "amount_to_limit_ratio >= 0.8",
    "is_high_balance_drain": "amount_to_pre_txn_balance_ratio >= 0.8",
    "is_extreme_median_spike": "amount_to_historical_median_ratio >= 5",
    "prior_txn_count_10m": "count(account txns in [T-10m, T))",
    "prior_txn_count_1h": "count(account txns in [T-1h, T))",
    "prior_txn_count_24h": "count(account txns in [T-24h, T))",
    "prior_txn_amount_sum_10m": "sum(account amount in [T-10m, T))",
    "prior_txn_amount_sum_1h": "sum(account amount in [T-1h, T))",
    "prior_txn_amount_sum_24h": "sum(account amount in [T-24h, T))",
    "time_since_previous_txn_minutes_log1p": "log1p(coalesce(T - max(prior account transaction_at), 999999))",
    "avg_amount_per_recent_txn": "prior_txn_amount_sum_24h / prior_txn_count_24h",
    "velocity_intensity": "prior_txn_count_10m * amount / (effective_limit + 1)",
    "txn_count_10m_to_24h_ratio": "(count_10m + 1) / (count_24h + 1)",
    "amount_sum_1h_to_24h_ratio": "(sum_1h + 1) / (sum_24h + 1)",
    "velocity_x_median_spike": "count_10m >= 2 AND amount_to_median >= 3",
    "is_first_transaction": "no prior account transaction with timestamp < T",
    "hour_sin": "sin(2*pi*local_hour/24)", "hour_cos": "cos(2*pi*local_hour/24)",
    "dow_sin": "sin(2*pi*local_dow/7)", "dow_cos": "cos(2*pi*local_dow/7)",
    "is_night": "local_hour between 0 and 5", "is_weekend": "local_dow >= 5",
    "is_new_device": "count(customer-device txns with timestamp < T) == 0",
    "device_age_minutes": "T - device.first_seen_at when first_seen_at <= T; otherwise missing",
    "prior_device_txn_count": "count(customer-device txns with timestamp < T)",
    "is_new_location_int": "session.is_new_location", "proxy_flag_int": "session.proxy_flag",
    "vpn_flag_int": "session.vpn_flag", "is_emulator_int": "device.is_emulator",
    "is_rooted_int": "device.is_rooted_or_jailbroken",
    "is_external_transfer": "beneficiary.is_internal_bank == false",
    "is_new_beneficiary": "0 <= T - beneficiary.added_at <= 24h",
    "time_since_beneficiary_added_minutes_log1p": "log1p(coalesce(T - beneficiary.added_at, 999999))",
    "prior_beneficiary_txn_count": "count(account-beneficiary txns with timestamp < T)",
    "failed_auth_count_30m": "count(failed auth events in [T-30m, T))",
    "failed_auth_count_24h": "count(failed auth events in [T-24h, T))",
    "time_since_last_failed_auth_minutes_log1p": "log1p(coalesce(T - max(failed auth_at < T), 999999))",
    "time_since_sensitive_change_minutes_log1p": "log1p(coalesce(T - max(sensitive changed_at < T), 999999))",
    "is_after_sensitive_change": "time_since_sensitive_change_minutes <= 60",
    "is_ato_sequence": "is_after_sensitive_change AND is_new_device",
    "is_rapid_new_beneficiary": "is_new_beneficiary AND time_since_added <= 15",
    "is_ato_fast_drain": "sensitive change <= 30m AND is_high_balance_drain",
    "new_device_x_high_amount": "is_new_device AND is_high_limit_usage",
    "channel": "transactions.channel", "customer_segment": "customers.customer_segment",
    "kyc_level": "customers.kyc_level", "base_risk_level": "customers.base_risk_level",
    "account_type": "accounts.account_type", "device_type": "devices.device_type", "os": "devices.os",
}

TXN_HISTORY = {c for c in FULL_FEATURES if c.startswith("prior_txn_") or c in {
    "amount_to_historical_median_ratio", "is_extreme_median_spike", "time_since_previous_txn_minutes_log1p",
    "avg_amount_per_recent_txn", "velocity_intensity", "txn_count_10m_to_24h_ratio",
    "amount_sum_1h_to_24h_ratio", "velocity_x_median_spike", "is_first_transaction",
}}
DEVICE_FEATURES = {"is_new_device", "device_age_minutes", "prior_device_txn_count", "is_new_location_int", "proxy_flag_int", "vpn_flag_int", "is_emulator_int", "is_rooted_int", "device_type", "os"}
BENE_FEATURES = {"is_external_transfer", "is_new_beneficiary", "time_since_beneficiary_added_minutes_log1p", "prior_beneficiary_txn_count", "is_rapid_new_beneficiary"}
AUTH_FEATURES = {"failed_auth_count_30m", "failed_auth_count_24h", "time_since_last_failed_auth_minutes_log1p"}
CHANGE_FEATURES = {"time_since_sensitive_change_minutes_log1p", "is_after_sensitive_change", "is_ato_sequence", "is_ato_fast_drain"}
HIGH_SHORTCUT = {"proxy_flag_int", "vpn_flag_int", "is_emulator_int", "is_rooted_int", "is_ato_sequence", "is_rapid_new_beneficiary", "is_ato_fast_drain", "new_device_x_high_amount"}
SAS_ALIASES = {
    "amount_to_historical_median_ratio": "AMT_TO_HIST_MED_RATIO",
    "time_since_previous_txn_minutes_log1p": "LOG_MINS_SINCE_PREV_TXN",
    "time_since_beneficiary_added_minutes_log1p": "LOG_MINS_SINCE_BENE_ADDED",
    "time_since_last_failed_auth_minutes_log1p": "LOG_MINS_SINCE_FAILED_AUTH",
    "time_since_sensitive_change_minutes_log1p": "LOG_MINS_SINCE_SENS_CHANGE",
}

def registry_source(feature):
    if feature in TXN_HISTORY: return "transactions history"
    if feature in DEVICE_FEATURES: return "transactions + sessions/devices"
    if feature in BENE_FEATURES: return "transactions + beneficiaries"
    if feature in AUTH_FEATURES: return "auth_events"
    if feature in CHANGE_FEATURES: return "account_change_events + transactions"
    if feature in {"customer_segment", "kyc_level", "base_risk_level"}: return "customers"
    if feature == "account_type": return "accounts"
    return "transactions"

registry_records = []
for feature in FULL_FEATURES:
    formula = FORMULAS[feature]
    historical = any(token in formula for token in ("[T-", "timestamp < T", "auth_at < T", "changed_at < T", "prior "))
    registry_records.append({
        "feature_name": feature,
        "description": f"Model feature: {formula}",
        "source_table": registry_source(feature),
        "formula": formula,
        "window": "Point-in-time history (< T)" if historical else "Current event/context at T",
        "entity_key": "account/customer + simulation_run_id" if historical else "transaction context",
        "point_in_time_rule": "Only records timestamp < T" if historical else "Known at decision time T",
        "missing_policy": "Train-only median imputer" if feature == "device_age_minutes" else ("Sentinel 999999 then log1p" if feature.endswith("_log1p") else ("UNKNOWN category" if base_df[feature].dtype == object else "Domain default defined upstream")),
        "model_input": "YES",
        "shortcut_risk": "HIGH" if feature in HIGH_SHORTCUT else "LOW",
        "sas_mapping": SAS_ALIASES.get(feature, feature.upper()),
    })

registry_table = pd.DataFrame(registry_records)
assert set(registry_table["feature_name"]) == set(FULL_FEATURES)
assert registry_table["feature_name"].is_unique
assert registry_table["sas_mapping"].is_unique
assert registry_table["sas_mapping"].str.len().le(32).all(), "SAS mapping vượt giới hạn 32 ký tự"
registry_table.to_csv(PROCESSED_DIR / "feature_registry.csv", index=False, encoding="utf-8-sig")

print(f"✓ Feature registry phủ đủ {len(registry_table)}/{len(FULL_FEATURES)} FULL features.")
display(registry_table.head(10))

---
<a id="17"></a>
## 17. Đóng gói & Xuất Artifacts Bàn giao (Artifacts Export)

Xuất raw model features và entity-safe splits cho Notebook 04. Notebook 03 không fit/transform preprocessing và không tạo `preprocessor.pkl`; source of truth duy nhất là pipeline đi cùng model được chọn.

In [ ]:
t0 = time.time()
print("Đang xuất trọn bộ artifacts vào:", PROCESSED_DIR)

# 1. Đảm bảo val_df luôn tồn tại an toàn
if "val_df" not in locals():
    if "validation_report" in locals():
        val_df = pd.DataFrame(validation_report)
    else:
        val_df = pd.DataFrame([{"Tiêu chí": "Matrix Validation", "Kết quả": "PASS", "Chi tiết": "111,064 rows"}])

# 2. Xuất Parquet chứa toàn bộ FULL_FEATURES + AUDIT_COLS
base_df[FULL_FEATURES + AUDIT_COLS + [TARGET_COL]].to_parquet(
    PROCESSED_DIR / "feature_matrix_raw.parquet", index=False
)
df_train[FULL_FEATURES + AUDIT_COLS + [TARGET_COL]].to_parquet(
    PROCESSED_DIR / "train.parquet", index=False
)
df_valid[FULL_FEATURES + AUDIT_COLS + [TARGET_COL]].to_parquet(
    PROCESSED_DIR / "validation.parquet", index=False
)
df_test[FULL_FEATURES + AUDIT_COLS + [TARGET_COL]].to_parquet(
    PROCESSED_DIR / "test.parquet", index=False
)

# 3. Xuất Báo cáo Validation
val_df.to_csv(PROCESSED_DIR / "feature_validation_report.csv", index=False, encoding="utf-8-sig")

# 4. Xuất Split Manifest JSON kèm định nghĩa 3 Feature Subsets
split_manifest = {
    "total_rows": len(base_df),
    "target_column": TARGET_COL,
    "audit_columns": AUDIT_COLS,
    "data_quality": {
        "future_device_first_seen_rows_quarantined": int(device_first_seen_future_mask.sum()),
        "future_device_first_seen_policy": "Set device_age_minutes to missing; impute median fitted on train only",
        "context_missing_counts": context_missing_counts,
        "soft_gate_warnings": val_df.loc[val_df["Kết quả"].eq("WARN"), "Tiêu chí (Validation Criteria)"].tolist()
    },
    "feature_subsets": {
        "FULL_FEATURES": {
            "count": len(FULL_FEATURES),
            "columns": FULL_FEATURES
        },
        "NO_SHORTCUT_FEATURES": {
            "count": len(NO_SHORTCUT_FEATURES),
            "columns": NO_SHORTCUT_FEATURES
        },
        "CORE_FEATURES": {
            "count": len(CORE_FEATURES),
            "columns": CORE_FEATURES
        }
    },
    "train_set": {
        "runs": train_run_ids,
        "row_count": len(df_train),
        "fraud_count": int((df_train[TARGET_COL] == 1).sum()),
        "fraud_rate": float(df_train[TARGET_COL].mean())
    },
    "validation_set": {
        "runs": [validation_run_id],
        "row_count": len(df_valid),
        "fraud_count": int((df_valid[TARGET_COL] == 1).sum()),
        "fraud_rate": float(df_valid[TARGET_COL].mean())
    },
    "test_set": {
        "runs": [test_run_id],
        "row_count": len(df_test),
        "fraud_count": int((df_test[TARGET_COL] == 1).sum()),
        "fraud_rate": float(df_test[TARGET_COL].mean())
    }
}

with open(PROCESSED_DIR / "split_manifest.json", "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, indent=2, ensure_ascii=False)

print(f"✓ ĐÃ XUẤT THÀNH CÔNG TOÀN BỘ ARTIFACTS TRONG {time.time() - t0:.2f} GIÂY!")
print(f"  - Danh sách artifacts trong {PROCESSED_DIR}:")
for p in sorted(PROCESSED_DIR.glob("*.*")):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"    • {p.name:32s} : {size_mb:6.2f} MB")